# Disease Prediction using Naive Bayes Classifier

This notebook implements a **Naive Bayes Multinomial Classifier** from scratch for disease prediction based on symptoms.

## Data Flow Architecture:
```
CSV Files (Assets)
       ↓
   CSVParser
       ↓
  TrainingData
       ↓
NaiveBayesClassifier.train()
       ↓
  Trained Model
       ↓
User Selects Symptoms
       ↓
NaiveBayesClassifier.predict()
       ↓
 PredictionResult
       ↓
    UI Display
```

## Step 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

## Step 2: CSVParser - Load and Parse Data from CSV Files

In [3]:
class CSVParser:
    """
    Parses CSV files containing symptom data and disease labels.
    """
    
    def __init__(self, file_path):
        """
        Initialize CSVParser with file path.
        
        Args:
            file_path (str): training.csv
        """
        self.file_path = file_path
        self.data = None
        self.symptoms = None
        self.diseases = None
    
    def load_data(self):
        """
        Load data from CSV file.
        
        Returns:
            pd.DataFrame: Loaded data
        """
        print(f"Loading data from: {self.file_path}")
        self.data = pd.read_csv(self.file_path)
        
        # Extract symptom columns (all except last column 'prognosis')
        self.symptoms = list(self.data.columns[:-1])
        
        # Extract unique diseases
        self.diseases = self.data['prognosis'].unique()
        
        print(f"Data loaded successfully!")
        print(f"Total samples: {len(self.data)}")
        print(f"Total symptoms: {len(self.symptoms)}")
        print(f"Total diseases: {len(self.diseases)}")
        
        return self.data
    
    def get_symptoms(self):
        """Get list of all symptoms."""
        return self.symptoms
    
    def get_diseases(self):
        """Get list of all diseases."""
        return self.diseases
    
    def get_data(self):
        """Get the loaded dataframe."""
        return self.data

# Initialize and load training data
csv_parser = CSVParser('training.csv')
training_dataframe = csv_parser.load_data()

Loading data from: training.csv
Data loaded successfully!
Total samples: 4920
Total symptoms: 132
Total diseases: 41


### Display Sample Data

In [4]:
print("\nFirst 5 rows of the dataset:")
print(training_dataframe.head())

print("\n\nDataset Info:")
print(f"Shape: {training_dataframe.shape}")
print(f"\nDisease Distribution:")
print(training_dataframe['prognosis'].value_counts())


First 5 rows of the dataset:
   itching  skin_rash  nodal_skin_eruptions  continuous_sneezing  shivering  \
0        1          1                     1                    0          0   
1        0          1                     1                    0          0   
2        1          0                     1                    0          0   
3        1          1                     0                    0          0   
4        1          1                     1                    0          0   

   chills  joint_pain  stomach_pain  acidity  ulcers_on_tongue  ...  \
0       0           0             0        0                 0  ...   
1       0           0             0        0                 0  ...   
2       0           0             0        0                 0  ...   
3       0           0             0        0                 0  ...   
4       0           0             0        0                 0  ...   

   blackheads  scurring  skin_peeling  silver_like_dusting  \
0     

## Step 3: TrainingData - Prepare Data for Training

In [5]:
class TrainingData:
    """
    Prepares and organizes training data for the classifier.
    """
    
    def __init__(self, dataframe, symptoms):
        """
        Initialize TrainingData.
        
        Args:
            dataframe (pd.DataFrame): Training data
            symptoms (list): List of symptom names
        """
        self.dataframe = dataframe
        self.symptoms = symptoms
        self.X = None  # Features (symptoms)
        self.y = None  # Labels (diseases)
        
    def prepare_data(self):
        """
        Split data into features (X) and labels (y).
        
        Returns:
            tuple: (X, y) - features and labels
        """
        # Extract features (all columns except 'prognosis')
        self.X = self.dataframe.iloc[:, :-1].values
        
        # Extract labels (disease names)
        self.y = self.dataframe['prognosis'].values
        
        print(f"Training data prepared:")
        print(f"  Features shape: {self.X.shape}")
        print(f"  Labels shape: {self.y.shape}")
        
        return self.X, self.y
    
    def get_feature_names(self):
        """Get symptom names."""
        return self.symptoms

# Prepare training data
training_data = TrainingData(training_dataframe, csv_parser.get_symptoms())
X_train, y_train = training_data.prepare_data()

Training data prepared:
  Features shape: (4920, 132)
  Labels shape: (4920,)


## Step 4: NaiveBayesClassifier - Implement Multinomial Naive Bayes

### Naive Bayes Theory:

**Naive Bayes** is a probabilistic classifier based on **Bayes' Theorem**:

$$P(Disease|Symptoms) = \frac{P(Symptoms|Disease) \times P(Disease)}{P(Symptoms)}$$

For classification, we need to find:

$$\text{Predicted Disease} = \arg\max_{disease} P(Disease|Symptoms)$$

Since $P(Symptoms)$ is constant for all diseases, we can simplify:

$$\text{Predicted Disease} = \arg\max_{disease} P(Symptoms|Disease) \times P(Disease)$$

**Multinomial Naive Bayes** assumes features are independent (naive assumption):

$$P(Symptoms|Disease) = \prod_{i=1}^{n} P(Symptom_i|Disease)$$

### Implementation Steps:
1. Calculate prior probabilities: $P(Disease)$
2. Calculate likelihood: $P(Symptom|Disease)$ for each symptom-disease pair
3. Use Laplace smoothing to avoid zero probabilities
4. For prediction, calculate posterior probability for each disease and select maximum

In [6]:
class NaiveBayesClassifier:
    """
    Multinomial Naive Bayes Classifier implementation from scratch.
    """
    
    def __init__(self, alpha=1.0):
        """
        Initialize Naive Bayes Classifier.
        
        Args:
            alpha (float): Laplace smoothing parameter (default=1.0)
        """
        self.alpha = alpha  # Smoothing parameter
        self.classes = None  # Unique disease classes
        self.class_priors = {}  # P(Disease)
        self.feature_probs = {}  # P(Symptom|Disease)
        self.num_features = None
        self.trained = False
        
    def train(self, X, y):
        """
        Train the Naive Bayes classifier.
        
        Args:
            X (np.array): Training features (symptoms) - shape (n_samples, n_features)
            y (np.array): Training labels (diseases) - shape (n_samples,)
        """
        print("\n" + "="*60)
        print("Training Naive Bayes Classifier")
        print("="*60)
        
        n_samples, self.num_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        
        print(f"Number of samples: {n_samples}")
        print(f"Number of features (symptoms): {self.num_features}")
        print(f"Number of classes (diseases): {n_classes}")
        
        # Calculate class priors: P(Disease)
        print("\n1. Calculating Prior Probabilities P(Disease)...")
        for cls in self.classes:
            # Count samples for this disease
            cls_count = np.sum(y == cls)
            # Calculate prior probability
            self.class_priors[cls] = cls_count / n_samples
        
        print(f"   Computed priors for {n_classes} diseases")
        
        # Calculate feature probabilities: P(Symptom|Disease)
        print("\n2. Calculating Likelihood P(Symptom|Disease)...")
        
        for cls in self.classes:
            # Get samples for this disease
            X_cls = X[y == cls]
            n_cls_samples = X_cls.shape[0]
            
            # Initialize probability dictionary for this class
            self.feature_probs[cls] = {}
            
            # For each feature (symptom)
            for feature_idx in range(self.num_features):
                # Count how many times this symptom appears in this disease
                symptom_count = np.sum(X_cls[:, feature_idx])
                
                # Apply Laplace smoothing: (count + alpha) / (total + alpha * 2)
                # We use 2 because each symptom can be 0 or 1 (binary)
                prob = (symptom_count + self.alpha) / (n_cls_samples + self.alpha * 2)
                
                self.feature_probs[cls][feature_idx] = {
                    'present': prob,  # P(Symptom=1|Disease)
                    'absent': 1 - prob  # P(Symptom=0|Disease)
                }
        
        print(f"   Computed likelihoods for {n_classes} diseases and {self.num_features} symptoms")
        
        self.trained = True
        print("\n" + "="*60)
        print("Training Complete!")
        print("="*60)
        
    def predict_proba(self, X):
        """
        Predict class probabilities for samples.
        
        Args:
            X (np.array): Features to predict - shape (n_samples, n_features)
            
        Returns:
            dict: Probabilities for each class
        """
        if not self.trained:
            raise Exception("Model must be trained before prediction!")
        
        # For single sample, reshape to 2D
        if len(X.shape) == 1:
            X = X.reshape(1, -1)
        
        predictions = []
        
        for sample in X:
            class_scores = {}
            
            # Calculate posterior probability for each disease
            for cls in self.classes:
                # Start with log of prior probability
                log_prob = np.log(self.class_priors[cls])
                
                # Multiply by likelihood of each symptom (in log space, we add)
                for feature_idx in range(self.num_features):
                    symptom_value = sample[feature_idx]
                    
                    # Get P(Symptom|Disease)
                    if symptom_value == 1:
                        prob = self.feature_probs[cls][feature_idx]['present']
                    else:
                        prob = self.feature_probs[cls][feature_idx]['absent']
                    
                    log_prob += np.log(prob)
                
                class_scores[cls] = log_prob
            
            predictions.append(class_scores)
        
        return predictions
    
    def predict(self, X):
        """
        Predict classes for samples.
        
        Args:
            X (np.array): Features to predict - shape (n_samples, n_features)
            
        Returns:
            list: Predicted disease names
        """
        # Get probabilities
        proba_list = self.predict_proba(X)
        
        # Select class with maximum probability
        predictions = []
        for class_scores in proba_list:
            predicted_class = max(class_scores, key=class_scores.get)
            predictions.append(predicted_class)
        
        return predictions
    
    def predict_with_probabilities(self, X, top_k=3):
        """
        Predict classes with probability scores.
        
        Args:
            X (np.array): Features to predict
            top_k (int): Number of top predictions to return
            
        Returns:
            list: List of dictionaries with disease and probability
        """
        proba_list = self.predict_proba(X)
        
        results = []
        for class_scores in proba_list:
            # Convert log probabilities to regular probabilities
            # Subtract max for numerical stability
            max_log_prob = max(class_scores.values())
            exp_scores = {cls: np.exp(log_prob - max_log_prob) 
                         for cls, log_prob in class_scores.items()}
            
            # Normalize to get probabilities
            total = sum(exp_scores.values())
            probabilities = {cls: score / total for cls, score in exp_scores.items()}
            
            # Sort by probability and get top k
            sorted_probs = sorted(probabilities.items(), key=lambda x: x[1], reverse=True)
            top_predictions = [
                {'disease': disease, 'probability': prob}
                for disease, prob in sorted_probs[:top_k]
            ]
            
            results.append(top_predictions)
        
        return results

# Initialize classifier
naive_bayes = NaiveBayesClassifier(alpha=1.0)

# Train the model
naive_bayes.train(X_train, y_train)


Training Naive Bayes Classifier
Number of samples: 4920
Number of features (symptoms): 132
Number of classes (diseases): 41

1. Calculating Prior Probabilities P(Disease)...
   Computed priors for 41 diseases

2. Calculating Likelihood P(Symptom|Disease)...
   Computed likelihoods for 41 diseases and 132 symptoms

Training Complete!


## Step 5: Model Evaluation - Test on Training Data

In [7]:
# Make predictions on training data
print("\nEvaluating model on training data...")
y_pred_train = naive_bayes.predict(X_train)

# Calculate accuracy
accuracy = np.mean(y_pred_train == y_train)
print(f"\n{'='*60}")
print(f"Training Accuracy: {accuracy * 100:.2f}%")
print(f"{'='*60}")

# Show some example predictions
print("\n\nSample Predictions (First 10):")
print("-" * 80)
print(f"{'Index':<8} {'Actual Disease':<25} {'Predicted Disease':<25} {'Match':<10}")
print("-" * 80)

for i in range(min(10, len(y_train))):
    match = "✓" if y_train[i] == y_pred_train[i] else "✗"
    print(f"{i:<8} {y_train[i]:<25} {y_pred_train[i]:<25} {match:<10}")

print("-" * 80)


Evaluating model on training data...

Training Accuracy: 100.00%


Sample Predictions (First 10):
--------------------------------------------------------------------------------
Index    Actual Disease            Predicted Disease         Match     
--------------------------------------------------------------------------------
0        Fungal infection          Fungal infection          ✓         
1        Fungal infection          Fungal infection          ✓         
2        Fungal infection          Fungal infection          ✓         
3        Fungal infection          Fungal infection          ✓         
4        Fungal infection          Fungal infection          ✓         
5        Fungal infection          Fungal infection          ✓         
6        Fungal infection          Fungal infection          ✓         
7        Fungal infection          Fungal infection          ✓         
8        Fungal infection          Fungal infection          ✓         
9        Fungal inf

## Step 6: Load Test Data

In [8]:
# Load test data
csv_parser_test = CSVParser('testing.csv')
test_dataframe = csv_parser_test.load_data()

# Prepare test data
test_data = TrainingData(test_dataframe, csv_parser_test.get_symptoms())
X_test, y_test = test_data.prepare_data()

Loading data from: testing.csv
Data loaded successfully!
Total samples: 41
Total symptoms: 132
Total diseases: 41
Training data prepared:
  Features shape: (41, 132)
  Labels shape: (41,)


## Step 7: Test Model on Test Data

In [9]:
# Make predictions on test data
print("\nEvaluating model on test data...")
y_pred_test = naive_bayes.predict(X_test)

# Calculate test accuracy
test_accuracy = np.mean(y_pred_test == y_test)
print(f"\n{'='*60}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"{'='*60}")

# Show some example predictions
print("\n\nSample Test Predictions (First 10):")
print("-" * 80)
print(f"{'Index':<8} {'Actual Disease':<25} {'Predicted Disease':<25} {'Match':<10}")
print("-" * 80)

for i in range(min(10, len(y_test))):
    match = "✓" if y_test[i] == y_pred_test[i] else "✗"
    print(f"{i:<8} {y_test[i]:<25} {y_pred_test[i]:<25} {match:<10}")

print("-" * 80)


Evaluating model on test data...

Test Accuracy: 100.00%


Sample Test Predictions (First 10):
--------------------------------------------------------------------------------
Index    Actual Disease            Predicted Disease         Match     
--------------------------------------------------------------------------------
0        Fungal infection          Fungal infection          ✓         
1        Allergy                   Allergy                   ✓         
2        GERD                      GERD                      ✓         
3        Chronic cholestasis       Chronic cholestasis       ✓         
4        Drug Reaction             Drug Reaction             ✓         
5        Peptic ulcer diseae       Peptic ulcer diseae       ✓         
6        AIDS                      AIDS                      ✓         
7        Diabetes                  Diabetes                  ✓         
8        Gastroenteritis           Gastroenteritis           ✓         
9        Bronchial Ast

## Step 8: PredictionResult - Interactive Prediction System

Now let's simulate user selecting symptoms and getting prediction results.

In [10]:
class PredictionResult:
    """
    Handles prediction results and displays them in a user-friendly format.
    """
    
    def __init__(self, classifier, symptoms_list):
        """
        Initialize PredictionResult.
        
        Args:
            classifier: Trained NaiveBayesClassifier
            symptoms_list (list): List of all symptom names
        """
        self.classifier = classifier
        self.symptoms_list = symptoms_list
        self.num_symptoms = len(symptoms_list)
    
    def predict_from_symptoms(self, selected_symptoms, top_k=3):
        """
        Predict disease based on selected symptoms.
        
        Args:
            selected_symptoms (list): List of symptom names that are present
            top_k (int): Number of top predictions to return
            
        Returns:
            dict: Prediction results
        """
        # Create feature vector
        feature_vector = np.zeros(self.num_symptoms)
        
        # Mark selected symptoms as 1
        for symptom in selected_symptoms:
            if symptom in self.symptoms_list:
                idx = self.symptoms_list.index(symptom)
                feature_vector[idx] = 1
        
        # Get predictions with probabilities
        predictions = self.classifier.predict_with_probabilities(
            feature_vector.reshape(1, -1), 
            top_k=top_k
        )[0]
        
        # Prepare result
        result = {
            'selected_symptoms': selected_symptoms,
            'num_symptoms': len(selected_symptoms),
            'predictions': predictions
        }
        
        return result
    
    def display_result(self, result):
        """
        Display prediction results in a formatted way.
        
        Args:
            result (dict): Result from predict_from_symptoms
        """
        print("\n" + "="*70)
        print(" "*20 + "DISEASE PREDICTION RESULT")
        print("="*70)
        
        print(f"\n📋 Selected Symptoms ({result['num_symptoms']}):")
        print("-" * 70)
        for i, symptom in enumerate(result['selected_symptoms'], 1):
            print(f"  {i}. {symptom}")
        
        print("\n" + "="*70)
        print("🏥 TOP PREDICTED DISEASES:")
        print("="*70)
        
        for i, pred in enumerate(result['predictions'], 1):
            disease = pred['disease']
            prob = pred['probability'] * 100
            
            # Create progress bar
            bar_length = 40
            filled = int(bar_length * pred['probability'])
            bar = "█" * filled + "░" * (bar_length - filled)
            
            print(f"\n{i}. {disease}")
            print(f"   Confidence: {prob:.2f}%")
            print(f"   [{bar}]")
        
        print("\n" + "="*70)
        print(f"🎯 Most Likely Disease: {result['predictions'][0]['disease']}")
        print(f"   Confidence: {result['predictions'][0]['probability'] * 100:.2f}%")
        print("="*70)

# Initialize PredictionResult
prediction_system = PredictionResult(naive_bayes, csv_parser.get_symptoms())

## Step 9: Example Predictions

Let's test the system with different symptom combinations.

In [11]:
# Example 1: Fungal Infection symptoms
print("\n\n" + "#"*70)
print("EXAMPLE 1: Fungal Infection Symptoms")
print("#"*70)

symptoms_1 = ['itching', 'skin_rash', 'nodal_skin_eruptions']
result_1 = prediction_system.predict_from_symptoms(symptoms_1, top_k=5)
prediction_system.display_result(result_1)



######################################################################
EXAMPLE 1: Fungal Infection Symptoms
######################################################################

                    DISEASE PREDICTION RESULT

📋 Selected Symptoms (3):
----------------------------------------------------------------------
  1. itching
  2. skin_rash
  3. nodal_skin_eruptions

🏥 TOP PREDICTED DISEASES:

1. Fungal infection
   Confidence: 99.99%
   [███████████████████████████████████████░]

2. Drug Reaction
   Confidence: 0.01%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

3. Acne
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

4. Impetigo
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

5. Allergy
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

🎯 Most Likely Disease: Fungal infection
   Confidence: 99.99%


In [12]:
# Example 2: Malaria symptoms
print("\n\n" + "#"*70)
print("EXAMPLE 2: Malaria Symptoms")
print("#"*70)

symptoms_2 = ['chills', 'vomiting', 'high_fever', 'sweating', 'headache', 'nausea', 'muscle_pain']
result_2 = prediction_system.predict_from_symptoms(symptoms_2, top_k=5)
prediction_system.display_result(result_2)



######################################################################
EXAMPLE 2: Malaria Symptoms
######################################################################

                    DISEASE PREDICTION RESULT

📋 Selected Symptoms (7):
----------------------------------------------------------------------
  1. chills
  2. vomiting
  3. high_fever
  4. sweating
  5. headache
  6. nausea
  7. muscle_pain

🏥 TOP PREDICTED DISEASES:

1. Malaria
   Confidence: 100.00%
   [███████████████████████████████████████░]

2. (vertigo) Paroymsal  Positional Vertigo
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

3. Typhoid
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

4. Heart attack
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

5. Paralysis (brain hemorrhage)
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

🎯 Most Likely Disease: Malaria
   Confidence: 100.00%


In [13]:
# Example 3: Diabetes symptoms
print("\n\n" + "#"*70)
print("EXAMPLE 3: Diabetes Symptoms")
print("#"*70)

symptoms_3 = ['fatigue', 'weight_loss', 'restlessness', 'lethargy', 
              'irregular_sugar_level', 'increased_appetite', 'polyuria']
result_3 = prediction_system.predict_from_symptoms(symptoms_3, top_k=5)
prediction_system.display_result(result_3)



######################################################################
EXAMPLE 3: Diabetes Symptoms
######################################################################

                    DISEASE PREDICTION RESULT

📋 Selected Symptoms (7):
----------------------------------------------------------------------
  1. fatigue
  2. weight_loss
  3. restlessness
  4. lethargy
  5. irregular_sugar_level
  6. increased_appetite
  7. polyuria

🏥 TOP PREDICTED DISEASES:

1. Diabetes 
   Confidence: 100.00%
   [███████████████████████████████████████░]

2. Jaundice
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

3. Hepatitis C
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

4. Allergy
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

5. Fungal infection
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

🎯 Most Likely Disease: Diabetes 
   Confidence: 100.00%


In [14]:
# Example 4: Common Cold/Allergy symptoms
print("\n\n" + "#"*70)
print("EXAMPLE 4: Common Cold/Allergy Symptoms")
print("#"*70)

symptoms_4 = ['continuous_sneezing', 'chills', 'watering_from_eyes']
result_4 = prediction_system.predict_from_symptoms(symptoms_4, top_k=5)
prediction_system.display_result(result_4)



######################################################################
EXAMPLE 4: Common Cold/Allergy Symptoms
######################################################################

                    DISEASE PREDICTION RESULT

📋 Selected Symptoms (3):
----------------------------------------------------------------------
  1. continuous_sneezing
  2. chills
  3. watering_from_eyes

🏥 TOP PREDICTED DISEASES:

1. Allergy
   Confidence: 100.00%
   [███████████████████████████████████████░]

2. Fungal infection
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

3. AIDS
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

4. Acne
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

5. Gastroenteritis
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

🎯 Most Likely Disease: Allergy
   Confidence: 100.00%


## Step 10: Complete Data Flow Demonstration

Let's demonstrate the complete data flow from start to finish.

In [15]:
def complete_data_flow_demo(selected_symptoms):
    """
    Demonstrates the complete data flow from CSV to prediction.
    """
    print("\n" + "█"*80)
    print("█" + " "*78 + "█")
    print("█" + " "*20 + "COMPLETE DATA FLOW DEMONSTRATION" + " "*26 + "█")
    print("█" + " "*78 + "█")
    print("█"*80)
    
    # Step 1: CSV Files
    print("\n📁 STEP 1: CSV Files (Assets)")
    print("   ├─ training.csv loaded ✓")
    print("   └─ testing.csv loaded ✓")
    
    # Step 2: CSVParser
    print("\n🔍 STEP 2: CSVParser")
    print(f"   ├─ Parsed {len(csv_parser.get_symptoms())} symptoms")
    print(f"   ├─ Parsed {len(csv_parser.get_diseases())} diseases")
    print(f"   └─ Total samples: {len(training_dataframe)}")
    
    # Step 3: TrainingData
    print("\n📊 STEP 3: TrainingData")
    print(f"   ├─ Features (X) shape: {X_train.shape}")
    print(f"   └─ Labels (y) shape: {y_train.shape}")
    
    # Step 4: NaiveBayesClassifier.train()
    print("\n🤖 STEP 4: NaiveBayesClassifier.train()")
    print(f"   ├─ Computed prior probabilities for {len(naive_bayes.classes)} diseases")
    print(f"   ├─ Computed likelihoods for {naive_bayes.num_features} symptoms")
    print(f"   └─ Training accuracy: {accuracy * 100:.2f}%")
    
    # Step 5: Trained Model
    print("\n✅ STEP 5: Trained Model")
    print("   ├─ Model is ready for predictions")
    print(f"   └─ Test accuracy: {test_accuracy * 100:.2f}%")
    
    # Step 6: User Selects Symptoms
    print("\n👤 STEP 6: User Selects Symptoms")
    print(f"   └─ Selected symptoms: {selected_symptoms}")
    
    # Step 7: NaiveBayesClassifier.predict()
    print("\n🔮 STEP 7: NaiveBayesClassifier.predict()")
    result = prediction_system.predict_from_symptoms(selected_symptoms, top_k=3)
    print(f"   ├─ Created feature vector with {result['num_symptoms']} active symptoms")
    print(f"   ├─ Calculated posterior probabilities for all diseases")
    print(f"   └─ Selected top {len(result['predictions'])} predictions")
    
    # Step 8: PredictionResult
    print("\n📋 STEP 8: PredictionResult")
    print("   └─ Formatted results for display:")
    for i, pred in enumerate(result['predictions'], 1):
        print(f"      {i}. {pred['disease']} ({pred['probability']*100:.2f}%)")
    
    # Step 9: UI Display
    print("\n🖥️  STEP 9: UI Display")
    print("   └─ Showing formatted prediction results...")
    
    prediction_system.display_result(result)
    
    print("\n" + "█"*80)
    print("█" + " "*25 + "DATA FLOW COMPLETE!" + " "*33 + "█")
    print("█"*80)

# Run the complete demo
complete_data_flow_demo(['high_fever', 'headache', 'nausea', 'vomiting', 'muscle_pain'])


████████████████████████████████████████████████████████████████████████████████
█                                                                              █
█                    COMPLETE DATA FLOW DEMONSTRATION                          █
█                                                                              █
████████████████████████████████████████████████████████████████████████████████

📁 STEP 1: CSV Files (Assets)
   ├─ training.csv loaded ✓
   └─ testing.csv loaded ✓

🔍 STEP 2: CSVParser
   ├─ Parsed 132 symptoms
   ├─ Parsed 41 diseases
   └─ Total samples: 4920

📊 STEP 3: TrainingData
   ├─ Features (X) shape: (4920, 132)
   └─ Labels (y) shape: (4920,)

🤖 STEP 4: NaiveBayesClassifier.train()
   ├─ Computed prior probabilities for 41 diseases
   ├─ Computed likelihoods for 132 symptoms
   └─ Training accuracy: 100.00%

✅ STEP 5: Trained Model
   ├─ Model is ready for predictions
   └─ Test accuracy: 100.00%

👤 STEP 6: User Selects Symptoms
   └─ Selected symptoms: 

## Step 11: Summary

### ✅ Complete Implementation of Naive Bayes Multinomial Classifier

We have successfully implemented:

1. **CSVParser**: Loads and parses symptom data from CSV files
2. **TrainingData**: Organizes data into features (X) and labels (y)
3. **NaiveBayesClassifier**: Complete from-scratch implementation with:
   - Prior probability calculation P(Disease)
   - Likelihood calculation P(Symptom|Disease)
   - Laplace smoothing for zero probability avoidance
   - Log probabilities for numerical stability
   - Prediction with confidence scores
4. **PredictionResult**: User-friendly result formatting
5. **UI Display**: Interactive prediction examples

### 📐 Mathematical Foundation:

$$P(Disease|Symptoms) \propto P(Symptoms|Disease) \times P(Disease)$$

### 🎯 Key Features:

- Direct implementation (no scikit-learn)
- Complete data flow from CSV to prediction
- High accuracy on training and test data
- Top-k predictions with confidence scores
- Beautiful visualizations

# Disease Prediction using Naive Bayes Classifier

This notebook implements a **Naive Bayes Multinomial Classifier** from scratch for disease prediction based on symptoms.

## Data Flow Architecture:
```
CSV Files (Assets)
       ↓
   CSVParser
       ↓
  TrainingData
       ↓
NaiveBayesClassifier.train()
       ↓
  Trained Model
       ↓
User Selects Symptoms
       ↓
NaiveBayesClassifier.predict()
       ↓
 PredictionResult
       ↓
    UI Display
```

## Step 1: Import Required Libraries

In [16]:
import pandas as pd
import numpy as np
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

## Step 2: CSVParser - Load and Parse Data from CSV Files

In [17]:
class CSVParser:
    """
    Parses CSV files containing symptom data and disease labels.
    """
    
    def __init__(self, file_path):
        """
        Initialize CSVParser with file path.
        
        Args:
            file_path (str): Path to the CSV file
        """
        self.file_path = file_path
        self.data = None
        self.symptoms = None
        self.diseases = None
    
    def load_data(self):
        """
        Load data from CSV file.
        
        Returns:
            pd.DataFrame: Loaded data
        """
        print(f"Loading data from: {self.file_path}")
        self.data = pd.read_csv(self.file_path)
        
        # Extract symptom columns (all except last column 'prognosis')
        self.symptoms = list(self.data.columns[:-1])
        
        # Extract unique diseases
        self.diseases = self.data['prognosis'].unique()
        
        print(f"Data loaded successfully!")
        print(f"Total samples: {len(self.data)}")
        print(f"Total symptoms: {len(self.symptoms)}")
        print(f"Total diseases: {len(self.diseases)}")
        
        return self.data
    
    def get_symptoms(self):
        """Get list of all symptoms."""
        return self.symptoms
    
    def get_diseases(self):
        """Get list of all diseases."""
        return self.diseases
    
    def get_data(self):
        """Get the loaded dataframe."""
        return self.data

# Initialize and load training data
csv_parser = CSVParser('training.csv')
training_dataframe = csv_parser.load_data()

Loading data from: training.csv
Data loaded successfully!
Total samples: 4920
Total symptoms: 132
Total diseases: 41


### Display Sample Data

In [18]:
print("\nFirst 5 rows of the dataset:")
print(training_dataframe.head())

print("\n\nDataset Info:")
print(f"Shape: {training_dataframe.shape}")
print(f"\nDisease Distribution:")
print(training_dataframe['prognosis'].value_counts())


First 5 rows of the dataset:
   itching  skin_rash  nodal_skin_eruptions  continuous_sneezing  shivering  \
0        1          1                     1                    0          0   
1        0          1                     1                    0          0   
2        1          0                     1                    0          0   
3        1          1                     0                    0          0   
4        1          1                     1                    0          0   

   chills  joint_pain  stomach_pain  acidity  ulcers_on_tongue  ...  \
0       0           0             0        0                 0  ...   
1       0           0             0        0                 0  ...   
2       0           0             0        0                 0  ...   
3       0           0             0        0                 0  ...   
4       0           0             0        0                 0  ...   

   blackheads  scurring  skin_peeling  silver_like_dusting  \
0     

## Step 3: TrainingData - Prepare Data for Training

In [19]:
class TrainingData:
    """
    Prepares and organizes training data for the classifier.
    """
    
    def __init__(self, dataframe, symptoms):
        """
        Initialize TrainingData.
        
        Args:
            dataframe (pd.DataFrame): Training data
            symptoms (list): List of symptom names
        """
        self.dataframe = dataframe
        self.symptoms = symptoms
        self.X = None  # Features (symptoms)
        self.y = None  # Labels (diseases)
        
    def prepare_data(self):
        """
        Split data into features (X) and labels (y).
        
        Returns:
            tuple: (X, y) - features and labels
        """
        # Extract features (all columns except 'prognosis')
        self.X = self.dataframe.iloc[:, :-1].values
        
        # Extract labels (disease names)
        self.y = self.dataframe['prognosis'].values
        
        print(f"Training data prepared:")
        print(f"  Features shape: {self.X.shape}")
        print(f"  Labels shape: {self.y.shape}")
        
        return self.X, self.y
    
    def get_feature_names(self):
        """Get symptom names."""
        return self.symptoms

# Prepare training data
training_data = TrainingData(training_dataframe, csv_parser.get_symptoms())
X_train, y_train = training_data.prepare_data()

Training data prepared:
  Features shape: (4920, 132)
  Labels shape: (4920,)


## Step 4: NaiveBayesClassifier - Implement Multinomial Naive Bayes

### Naive Bayes Theory:

**Naive Bayes** is a probabilistic classifier based on **Bayes' Theorem**:

$$P(Disease|Symptoms) = \frac{P(Symptoms|Disease) \times P(Disease)}{P(Symptoms)}$$

For classification, we need to find:

$$\text{Predicted Disease} = \arg\max_{disease} P(Disease|Symptoms)$$

Since $P(Symptoms)$ is constant for all diseases, we can simplify:

$$\text{Predicted Disease} = \arg\max_{disease} P(Symptoms|Disease) \times P(Disease)$$

**Multinomial Naive Bayes** assumes features are independent (naive assumption):

$$P(Symptoms|Disease) = \prod_{i=1}^{n} P(Symptom_i|Disease)$$

### Implementation Steps:
1. Calculate prior probabilities: $P(Disease)$
2. Calculate likelihood: $P(Symptom|Disease)$ for each symptom-disease pair
3. Use Laplace smoothing to avoid zero probabilities
4. For prediction, calculate posterior probability for each disease and select maximum

In [ ]:
class NaiveBayesClassifier:
    """
    Multinomial Naive Bayes Classifier implementation from scratch.
    """
    
    def __init__(self, alpha=1.0):
        """
        Initialize Naive Bayes Classifier.
        
        Args:
            alpha (float): Laplace smoothing parameter (default=1.0)
        """
        self.alpha = alpha  # Smoothing parameter
        self.classes = None  # Unique disease classes
        self.class_priors = {}  # P(Disease)
        self.feature_probs = {}  # P(Symptom|Disease)
        self.num_features = None
        self.trained = False
        
    def train(self, X, y):
        """
        Train the Naive Bayes classifier.
        
        Args:
            X (np.array): Training features (symptoms) - shape (n_samples, n_features)
            y (np.array): Training labels (diseases) - shape (n_samples,)
        """
        print("\n" + "="*60)
        print("Training Naive Bayes Classifier")
        print("="*60)
        
        n_samples, self.num_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        
        print(f"Number of samples: {n_samples}")
        print(f"Number of features (symptoms): {self.num_features}")
        print(f"Number of classes (diseases): {n_classes}")
        
        # Calculate class priors: P(Disease)
        print("\n1. Calculating Prior Probabilities P(Disease)...")
        for cls in self.classes:
            # Count samples for this disease
            cls_count = np.sum(y == cls)
            # Calculate prior probability
            self.class_priors[cls] = cls_count / n_samples
        
        print(f"   Computed priors for {n_classes} diseases")
        
        # Calculate feature probabilities: P(Symptom|Disease)
        print("\n2. Calculating Likelihood P(Symptom|Disease)...")
        
        for cls in self.classes:
            # Get samples for this disease
            X_cls = X[y == cls]
            n_cls_samples = X_cls.shape[0]
            
            # Initialize probability dictionary for this class
            self.feature_probs[cls] = {}
            
            # For each feature (symptom)
            for feature_idx in range(self.num_features):
                # Count how many times this symptom appears in this disease
                symptom_count = np.sum(X_cls[:, feature_idx])
                
                # Apply Laplace smoothing: (count + alpha) / (total + alpha * 2)
                # We use 2 because each symptom can be 0 or 1 (binary)
                prob = (symptom_count + self.alpha) / (n_cls_samples + self.alpha * 2)
                
                self.feature_probs[cls][feature_idx] = {
                    'present': prob,  # P(Symptom=1|Disease)
                    'absent': 1 - prob  # P(Symptom=0|Disease)
                }
        
        print(f"   Computed likelihoods for {n_classes} diseases and {self.num_features} symptoms")
        
        self.trained = True
        print("\n" + "="*60)
        print("Training Complete!")
        print("="*60)
        
    def predict_proba(self, X):
        """
        Predict class probabilities for samples.
        
        Args:
            X (np.array): Features to predict - shape (n_samples, n_features)
            
        Returns:
            dict: Probabilities for each class
        """
        if not self.trained:
            raise Exception("Model must be trained before prediction!")
        
        # For single sample, reshape to 2D
        if len(X.shape) == 1:
            X = X.reshape(1, -1)
        
        predictions = []
        
        for sample in X:
            class_scores = {}
            
            # Calculate posterior probability for each disease
            for cls in self.classes:
                # Start with log of prior probability
                log_prob = np.log(self.class_priors[cls])
                
                # Multiply by likelihood of each symptom (in log space, we add)
                for feature_idx in range(self.num_features):
                    symptom_value = sample[feature_idx]
                    
                    # Get P(Symptom|Disease)
                    if symptom_value == 1:
                        prob = self.feature_probs[cls][feature_idx]['present']
                    else:
                        prob = self.feature_probs[cls][feature_idx]['absent']
                    
                    log_prob += np.log(prob)
                
                class_scores[cls] = log_prob
            
            predictions.append(class_scores)
        
        return predictions
    
    def predict(self, X):
        """
        Predict classes for samples.
        
        Args:
            X (np.array): Features to predict - shape (n_samples, n_features)
            
        Returns:
            list: Predicted disease names
        """
        # Get probabilities
        proba_list = self.predict_proba(X)
        
        # Select class with maximum probability
        predictions = []
        for class_scores in proba_list:
            predicted_class = max(class_scores, key=class_scores.get)
            predictions.append(predicted_class)
        
        return predictions
    
    def predict_with_probabilities(self, X, top_k=3):
        """
        Predict classes with probability scores.
        
        Args:
            X (np.array): Features to predict
            top_k (int): Number of top predictions to return
            
        Returns:
            list: List of dictionaries with disease and probability
        """
        proba_list = self.predict_proba(X)
        
        results = []
        for class_scores in proba_list:
            # Convert log probabilities to regular probabilities
            # Subtract max for numerical stability
            max_log_prob = max(class_scores.values())
            exp_scores = {cls: np.exp(log_prob - max_log_prob) 
                         for cls, log_prob in class_scores.items()}
            
            # Normalize to get probabilities
            total = sum(exp_scores.values())
            probabilities = {cls: score / total for cls, score in exp_scores.items()}
            
            # Sort by probability and get top k
            sorted_probs = sorted(probabilities.items(), key=lambda x: x[1], reverse=True)
            top_predictions = [
                {'disease': disease, 'probability': prob}
                for disease, prob in sorted_probs[:top_k]
            ]
            
            results.append(top_predictions)
        
        return results

# Initialize classifier
naive_bayes = NaiveBayesClassifier(alpha=1.0)

# Train the model
naive_bayes.train(X_train, y_train)

## Step 5: Model Evaluation - Test on Training Data

In [20]:
# Make predictions on training data
print("\nEvaluating model on training data...")
y_pred_train = naive_bayes.predict(X_train)

# Calculate accuracy
accuracy = np.mean(y_pred_train == y_train)
print(f"\n{'='*60}")
print(f"Training Accuracy: {accuracy * 100:.2f}%")
print(f"{'='*60}")

# Show some example predictions
print("\n\nSample Predictions (First 10):")
print("-" * 80)
print(f"{'Index':<8} {'Actual Disease':<25} {'Predicted Disease':<25} {'Match':<10}")
print("-" * 80)

for i in range(min(10, len(y_train))):
    match = "✓" if y_train[i] == y_pred_train[i] else "✗"
    print(f"{i:<8} {y_train[i]:<25} {y_pred_train[i]:<25} {match:<10}")

print("-" * 80)


Evaluating model on training data...

Training Accuracy: 100.00%


Sample Predictions (First 10):
--------------------------------------------------------------------------------
Index    Actual Disease            Predicted Disease         Match     
--------------------------------------------------------------------------------
0        Fungal infection          Fungal infection          ✓         
1        Fungal infection          Fungal infection          ✓         
2        Fungal infection          Fungal infection          ✓         
3        Fungal infection          Fungal infection          ✓         
4        Fungal infection          Fungal infection          ✓         
5        Fungal infection          Fungal infection          ✓         
6        Fungal infection          Fungal infection          ✓         
7        Fungal infection          Fungal infection          ✓         
8        Fungal infection          Fungal infection          ✓         
9        Fungal inf

## Step 6: Load Test Data

In [21]:
# Load test data
csv_parser_test = CSVParser('testing.csv')
test_dataframe = csv_parser_test.load_data()

# Prepare test data
test_data = TrainingData(test_dataframe, csv_parser_test.get_symptoms())
X_test, y_test = test_data.prepare_data()

Loading data from: testing.csv
Data loaded successfully!
Total samples: 41
Total symptoms: 132
Total diseases: 41
Training data prepared:
  Features shape: (41, 132)
  Labels shape: (41,)


## Step 7: Test Model on Test Data

In [22]:
# Make predictions on test data
print("\nEvaluating model on test data...")
y_pred_test = naive_bayes.predict(X_test)

# Calculate test accuracy
test_accuracy = np.mean(y_pred_test == y_test)
print(f"\n{'='*60}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"{'='*60}")

# Show some example predictions
print("\n\nSample Test Predictions (First 10):")
print("-" * 80)
print(f"{'Index':<8} {'Actual Disease':<25} {'Predicted Disease':<25} {'Match':<10}")
print("-" * 80)

for i in range(min(10, len(y_test))):
    match = "✓" if y_test[i] == y_pred_test[i] else "✗"
    print(f"{i:<8} {y_test[i]:<25} {y_pred_test[i]:<25} {match:<10}")

print("-" * 80)


Evaluating model on test data...

Test Accuracy: 100.00%


Sample Test Predictions (First 10):
--------------------------------------------------------------------------------
Index    Actual Disease            Predicted Disease         Match     
--------------------------------------------------------------------------------
0        Fungal infection          Fungal infection          ✓         
1        Allergy                   Allergy                   ✓         
2        GERD                      GERD                      ✓         
3        Chronic cholestasis       Chronic cholestasis       ✓         
4        Drug Reaction             Drug Reaction             ✓         
5        Peptic ulcer diseae       Peptic ulcer diseae       ✓         
6        AIDS                      AIDS                      ✓         
7        Diabetes                  Diabetes                  ✓         
8        Gastroenteritis           Gastroenteritis           ✓         
9        Bronchial Ast

## Step 8: PredictionResult - Interactive Prediction System

Now let's simulate user selecting symptoms and getting prediction results.

In [23]:
class PredictionResult:
    """
    Handles prediction results and displays them in a user-friendly format.
    """
    
    def __init__(self, classifier, symptoms_list):
        """
        Initialize PredictionResult.
        
        Args:
            classifier: Trained NaiveBayesClassifier
            symptoms_list (list): List of all symptom names
        """
        self.classifier = classifier
        self.symptoms_list = symptoms_list
        self.num_symptoms = len(symptoms_list)
    
    def predict_from_symptoms(self, selected_symptoms, top_k=3):
        """
        Predict disease based on selected symptoms.
        
        Args:
            selected_symptoms (list): List of symptom names that are present
            top_k (int): Number of top predictions to return
            
        Returns:
            dict: Prediction results
        """
        # Create feature vector
        feature_vector = np.zeros(self.num_symptoms)
        
        # Mark selected symptoms as 1
        for symptom in selected_symptoms:
            if symptom in self.symptoms_list:
                idx = self.symptoms_list.index(symptom)
                feature_vector[idx] = 1
        
        # Get predictions with probabilities
        predictions = self.classifier.predict_with_probabilities(
            feature_vector.reshape(1, -1), 
            top_k=top_k
        )[0]
        
        # Prepare result
        result = {
            'selected_symptoms': selected_symptoms,
            'num_symptoms': len(selected_symptoms),
            'predictions': predictions
        }
        
        return result
    
    def display_result(self, result):
        """
        Display prediction results in a formatted way.
        
        Args:
            result (dict): Result from predict_from_symptoms
        """
        print("\n" + "="*70)
        print(" "*20 + "DISEASE PREDICTION RESULT")
        print("="*70)
        
        print(f"\n📋 Selected Symptoms ({result['num_symptoms']}):")
        print("-" * 70)
        for i, symptom in enumerate(result['selected_symptoms'], 1):
            print(f"  {i}. {symptom}")
        
        print("\n" + "="*70)
        print("🏥 TOP PREDICTED DISEASES:")
        print("="*70)
        
        for i, pred in enumerate(result['predictions'], 1):
            disease = pred['disease']
            prob = pred['probability'] * 100
            
            # Create progress bar
            bar_length = 40
            filled = int(bar_length * pred['probability'])
            bar = "█" * filled + "░" * (bar_length - filled)
            
            print(f"\n{i}. {disease}")
            print(f"   Confidence: {prob:.2f}%")
            print(f"   [{bar}]")
        
        print("\n" + "="*70)
        print(f"🎯 Most Likely Disease: {result['predictions'][0]['disease']}")
        print(f"   Confidence: {result['predictions'][0]['probability'] * 100:.2f}%")
        print("="*70)

# Initialize PredictionResult
prediction_system = PredictionResult(naive_bayes, csv_parser.get_symptoms())

## Step 9: UI Display - Example Predictions

Let's test the system with different symptom combinations.

In [24]:
# Example 1: Fungal Infection symptoms
print("\n\n" + "#"*70)
print("EXAMPLE 1: Fungal Infection Symptoms")
print("#"*70)

symptoms_1 = ['itching', 'skin_rash', 'nodal_skin_eruptions']
result_1 = prediction_system.predict_from_symptoms(symptoms_1, top_k=5)
prediction_system.display_result(result_1)



######################################################################
EXAMPLE 1: Fungal Infection Symptoms
######################################################################

                    DISEASE PREDICTION RESULT

📋 Selected Symptoms (3):
----------------------------------------------------------------------
  1. itching
  2. skin_rash
  3. nodal_skin_eruptions

🏥 TOP PREDICTED DISEASES:

1. Fungal infection
   Confidence: 99.99%
   [███████████████████████████████████████░]

2. Drug Reaction
   Confidence: 0.01%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

3. Acne
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

4. Impetigo
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

5. Allergy
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

🎯 Most Likely Disease: Fungal infection
   Confidence: 99.99%


In [25]:
# Example 2: Malaria symptoms
print("\n\n" + "#"*70)
print("EXAMPLE 2: Malaria Symptoms")
print("#"*70)

symptoms_2 = ['chills', 'vomiting', 'high_fever', 'sweating', 'headache', 'nausea', 'muscle_pain']
result_2 = prediction_system.predict_from_symptoms(symptoms_2, top_k=5)
prediction_system.display_result(result_2)



######################################################################
EXAMPLE 2: Malaria Symptoms
######################################################################

                    DISEASE PREDICTION RESULT

📋 Selected Symptoms (7):
----------------------------------------------------------------------
  1. chills
  2. vomiting
  3. high_fever
  4. sweating
  5. headache
  6. nausea
  7. muscle_pain

🏥 TOP PREDICTED DISEASES:

1. Malaria
   Confidence: 100.00%
   [███████████████████████████████████████░]

2. (vertigo) Paroymsal  Positional Vertigo
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

3. Typhoid
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

4. Heart attack
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

5. Paralysis (brain hemorrhage)
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

🎯 Most Likely Disease: Malaria
   Confidence: 100.00%


In [26]:
# Example 3: Diabetes symptoms
print("\n\n" + "#"*70)
print("EXAMPLE 3: Diabetes Symptoms")
print("#"*70)

symptoms_3 = ['fatigue', 'weight_loss', 'restlessness', 'lethargy', 
              'irregular_sugar_level', 'increased_appetite', 'polyuria']
result_3 = prediction_system.predict_from_symptoms(symptoms_3, top_k=5)
prediction_system.display_result(result_3)



######################################################################
EXAMPLE 3: Diabetes Symptoms
######################################################################

                    DISEASE PREDICTION RESULT

📋 Selected Symptoms (7):
----------------------------------------------------------------------
  1. fatigue
  2. weight_loss
  3. restlessness
  4. lethargy
  5. irregular_sugar_level
  6. increased_appetite
  7. polyuria

🏥 TOP PREDICTED DISEASES:

1. Diabetes 
   Confidence: 100.00%
   [███████████████████████████████████████░]

2. Jaundice
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

3. Hepatitis C
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

4. Allergy
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

5. Fungal infection
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

🎯 Most Likely Disease: Diabetes 
   Confidence: 100.00%


In [27]:
# Example 4: Common Cold/Allergy symptoms
print("\n\n" + "#"*70)
print("EXAMPLE 4: Common Cold/Allergy Symptoms")
print("#"*70)

symptoms_4 = ['continuous_sneezing', 'chills', 'watering_from_eyes']
result_4 = prediction_system.predict_from_symptoms(symptoms_4, top_k=5)
prediction_system.display_result(result_4)



######################################################################
EXAMPLE 4: Common Cold/Allergy Symptoms
######################################################################

                    DISEASE PREDICTION RESULT

📋 Selected Symptoms (3):
----------------------------------------------------------------------
  1. continuous_sneezing
  2. chills
  3. watering_from_eyes

🏥 TOP PREDICTED DISEASES:

1. Allergy
   Confidence: 100.00%
   [███████████████████████████████████████░]

2. Fungal infection
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

3. AIDS
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

4. Acne
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

5. Gastroenteritis
   Confidence: 0.00%
   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

🎯 Most Likely Disease: Allergy
   Confidence: 100.00%


## Step 10: Visualize Model Parameters

Let's examine some of the learned probabilities.

In [28]:
# Display class priors
print("\n" + "="*70)
print("CLASS PRIOR PROBABILITIES P(Disease)")
print("="*70)

sorted_priors = sorted(naive_bayes.class_priors.items(), 
                       key=lambda x: x[1], reverse=True)

print(f"\n{'Disease':<30} {'Prior Probability':<20} {'Percentage':<15}")
print("-"*70)

for disease, prior in sorted_priors[:10]:  # Show top 10
    print(f"{disease:<30} {prior:<20.6f} {prior*100:<15.2f}%")

print(f"\n... and {len(sorted_priors) - 10} more diseases")
print("-"*70)


CLASS PRIOR PROBABILITIES P(Disease)

Disease                        Prior Probability    Percentage     
----------------------------------------------------------------------
(vertigo) Paroymsal  Positional Vertigo 0.024390             2.44           %
AIDS                           0.024390             2.44           %
Acne                           0.024390             2.44           %
Alcoholic hepatitis            0.024390             2.44           %
Allergy                        0.024390             2.44           %
Arthritis                      0.024390             2.44           %
Bronchial Asthma               0.024390             2.44           %
Cervical spondylosis           0.024390             2.44           %
Chicken pox                    0.024390             2.44           %
Chronic cholestasis            0.024390             2.44           %

... and 31 more diseases
----------------------------------------------------------------------


In [29]:
# Display feature probabilities for a specific disease
print("\n" + "="*70)
print("FEATURE PROBABILITIES FOR 'Fungal infection'")
print("="*70)

disease = 'Fungal infection'
symptom_probs = []

for idx, symptom in enumerate(csv_parser.get_symptoms()[:20]):  # Show first 20 symptoms
    prob_present = naive_bayes.feature_probs[disease][idx]['present']
    symptom_probs.append((symptom, prob_present))

# Sort by probability
symptom_probs.sort(key=lambda x: x[1], reverse=True)

print(f"\n{'Symptom':<35} {'P(Symptom=1|Disease)':<25}")
print("-"*70)

for symptom, prob in symptom_probs[:15]:  # Top 15
    bar_length = 30
    filled = int(bar_length * prob)
    bar = "█" * filled + "░" * (bar_length - filled)
    print(f"{symptom:<35} {prob:.4f} [{bar}]")

print("-"*70)


FEATURE PROBABILITIES FOR 'Fungal infection'

Symptom                             P(Symptom=1|Disease)     
----------------------------------------------------------------------
itching                             0.8934 [██████████████████████████░░░░]
skin_rash                           0.8934 [██████████████████████████░░░░]
nodal_skin_eruptions                0.8934 [██████████████████████████░░░░]
continuous_sneezing                 0.0082 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
shivering                           0.0082 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
chills                              0.0082 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
joint_pain                          0.0082 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
stomach_pain                        0.0082 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
acidity                             0.0082 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
ulcers_on_tongue                    0.0082 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
muscle_wasting                      0.0082 [░░░░░░░░░░░░░░░░

## Step 11: Complete Data Flow Demonstration

Let's demonstrate the complete data flow from start to finish.

In [30]:
def complete_data_flow_demo(selected_symptoms):
    """
    Demonstrates the complete data flow from CSV to prediction.
    """
    print("\n" + "█"*80)
    print("█" + " "*78 + "█")
    print("█" + " "*20 + "COMPLETE DATA FLOW DEMONSTRATION" + " "*26 + "█")
    print("█" + " "*78 + "█")
    print("█"*80)
    
    # Step 1: CSV Files
    print("\n📁 STEP 1: CSV Files (Assets)")
    print("   ├─ training.csv loaded ✓")
    print("   └─ testing.csv loaded ✓")
    
    # Step 2: CSVParser
    print("\n🔍 STEP 2: CSVParser")
    print(f"   ├─ Parsed {len(csv_parser.get_symptoms())} symptoms")
    print(f"   ├─ Parsed {len(csv_parser.get_diseases())} diseases")
    print(f"   └─ Total samples: {len(training_dataframe)}")
    
    # Step 3: TrainingData
    print("\n📊 STEP 3: TrainingData")
    print(f"   ├─ Features (X) shape: {X_train.shape}")
    print(f"   └─ Labels (y) shape: {y_train.shape}")
    
    # Step 4: NaiveBayesClassifier.train()
    print("\n🤖 STEP 4: NaiveBayesClassifier.train()")
    print(f"   ├─ Computed prior probabilities for {len(naive_bayes.classes)} diseases")
    print(f"   ├─ Computed likelihoods for {naive_bayes.num_features} symptoms")
    print(f"   └─ Training accuracy: {accuracy * 100:.2f}%")
    
    # Step 5: Trained Model
    print("\n✅ STEP 5: Trained Model")
    print("   ├─ Model is ready for predictions")
    print(f"   └─ Test accuracy: {test_accuracy * 100:.2f}%")
    
    # Step 6: User Selects Symptoms
    print("\n👤 STEP 6: User Selects Symptoms")
    print(f"   └─ Selected symptoms: {selected_symptoms}")
    
    # Step 7: NaiveBayesClassifier.predict()
    print("\n🔮 STEP 7: NaiveBayesClassifier.predict()")
    result = prediction_system.predict_from_symptoms(selected_symptoms, top_k=3)
    print(f"   ├─ Created feature vector with {result['num_symptoms']} active symptoms")
    print(f"   ├─ Calculated posterior probabilities for all diseases")
    print(f"   └─ Selected top {len(result['predictions'])} predictions")
    
    # Step 8: PredictionResult
    print("\n📋 STEP 8: PredictionResult")
    print("   └─ Formatted results for display:")
    for i, pred in enumerate(result['predictions'], 1):
        print(f"      {i}. {pred['disease']} ({pred['probability']*100:.2f}%)")
    
    # Step 9: UI Display
    print("\n🖥️  STEP 9: UI Display")
    print("   └─ Showing formatted prediction results...")
    
    prediction_system.display_result(result)
    
    print("\n" + "█"*80)
    print("█" + " "*25 + "DATA FLOW COMPLETE!" + " "*33 + "█")
    print("█"*80)

# Run the complete demo
complete_data_flow_demo(['high_fever', 'headache', 'nausea', 'vomiting', 'muscle_pain'])


████████████████████████████████████████████████████████████████████████████████
█                                                                              █
█                    COMPLETE DATA FLOW DEMONSTRATION                          █
█                                                                              █
████████████████████████████████████████████████████████████████████████████████

📁 STEP 1: CSV Files (Assets)
   ├─ training.csv loaded ✓
   └─ testing.csv loaded ✓

🔍 STEP 2: CSVParser
   ├─ Parsed 132 symptoms
   ├─ Parsed 41 diseases
   └─ Total samples: 4920

📊 STEP 3: TrainingData
   ├─ Features (X) shape: (4920, 132)
   └─ Labels (y) shape: (4920,)

🤖 STEP 4: NaiveBayesClassifier.train()
   ├─ Computed prior probabilities for 41 diseases
   ├─ Computed likelihoods for 132 symptoms
   └─ Training accuracy: 100.00%

✅ STEP 5: Trained Model
   ├─ Model is ready for predictions
   └─ Test accuracy: 100.00%

👤 STEP 6: User Selects Symptoms
   └─ Selected symptoms: 

## Step 12: Summary and Key Insights

### What We Implemented:

1. **CSVParser**: Loads and parses symptom data from CSV files
2. **TrainingData**: Organizes data into features (X) and labels (y)
3. **NaiveBayesClassifier**: Complete implementation of Multinomial Naive Bayes from scratch
   - Computes prior probabilities P(Disease)
   - Computes likelihood P(Symptom|Disease) with Laplace smoothing
   - Uses log probabilities to avoid numerical underflow
   - Predicts disease based on Bayes' theorem
4. **PredictionResult**: Formats predictions with confidence scores
5. **UI Display**: User-friendly visualization of results

### Key Mathematical Concepts:

- **Bayes' Theorem**: $P(Disease|Symptoms) \propto P(Symptoms|Disease) \times P(Disease)$
- **Naive Assumption**: Symptoms are independent given the disease
- **Laplace Smoothing**: Prevents zero probabilities
- **Log Probabilities**: Prevents numerical underflow

### Model Performance:

- High accuracy on both training and test data
- Provides probability estimates for top predictions
- Handles multiple symptoms effectively

In [31]:
# Final summary statistics
print("\n" + "="*80)
print(" "*25 + "NAIVE BAYES CLASSIFIER SUMMARY")
print("="*80)

print(f"\n📊 Dataset Statistics:")
print(f"   • Training samples: {len(X_train)}")
print(f"   • Test samples: {len(X_test)}")
print(f"   • Number of symptoms: {len(csv_parser.get_symptoms())}")
print(f"   • Number of diseases: {len(csv_parser.get_diseases())}")

print(f"\n🎯 Model Performance:")
print(f"   • Training Accuracy: {accuracy * 100:.2f}%")
print(f"   • Test Accuracy: {test_accuracy * 100:.2f}%")

print(f"\n⚙️ Model Parameters:")
print(f"   • Smoothing parameter (alpha): {naive_bayes.alpha}")
print(f"   • Total parameters learned: {len(naive_bayes.classes) * (1 + 2 * naive_bayes.num_features)}")

print(f"\n✨ Unique Diseases in Dataset:")
diseases = csv_parser.get_diseases()
for i in range(0, min(len(diseases), 20), 4):
    row_diseases = diseases[i:i+4]
    print(f"   {' | '.join([f'{d:<20}' for d in row_diseases])}")

if len(diseases) > 20:
    print(f"   ... and {len(diseases) - 20} more diseases")

print("\n" + "="*80)
print("✅ Naive Bayes implementation complete!")
print("="*80)


                         NAIVE BAYES CLASSIFIER SUMMARY

📊 Dataset Statistics:
   • Training samples: 4920
   • Test samples: 41
   • Number of symptoms: 132
   • Number of diseases: 41

🎯 Model Performance:
   • Training Accuracy: 100.00%
   • Test Accuracy: 100.00%

⚙️ Model Parameters:
   • Smoothing parameter (alpha): 1.0
   • Total parameters learned: 10865

✨ Unique Diseases in Dataset:
   Fungal infection     | Allergy              | GERD                 | Chronic cholestasis 
   Drug Reaction        | Peptic ulcer diseae  | AIDS                 | Diabetes            
   Gastroenteritis      | Bronchial Asthma     | Hypertension         | Migraine            
   Cervical spondylosis | Paralysis (brain hemorrhage) | Jaundice             | Malaria             
   Chicken pox          | Dengue               | Typhoid              | hepatitis A         
   ... and 21 more diseases

✅ Naive Bayes implementation complete!
